# Linear Regression

## Introducción

> [Linear Regression and Ordinary Least Squares | DotCSV](https://www.youtube.com/watch?v=k964_uNn3l0)


La **Linear Regression** es un método estadístico que nos permite modelar la relación entre una **variable dependiente** $Y$ y una o más **variables independientes** $X_i$ (también llamadas **predictores** o **features**). El objetivo es encontrar una **función lineal** $f$ que nos permita predecir el valor de $Y$ para cualquier valor de $X_i$.

Para simplificar y entender los conceptos básicos, en este ejemplo crearemos un modelo que predice el salario de un empleado considerando únicamente sus años de experiencia. Es decir, usaremos una única variable independiente, lo que significa que estamos ante un caso de **simple linear regression**. Es evidente que sería necesario analizar más variables para tener un mejor modelo. Si quisiéramos considerar más variables independientes (p. ej., nivel educativo, tipo de trabajo, sector industrial, ciudad, etc.), necesitaríamos usar un modelo de **multiple linear regression**.

## Lectura y exploración de datos

Tenemos datos sobre los salarios de empleados de una empresa junto con los años de experiencia de cada empleado (archivo `salaries.csv`). Usamos la librería `pandas` para leer los datos de un archivo [CSV](https://en.wikipedia.org/wiki/Comma-separated_values) y explorarlos.

In [ ]:
import pandas as pd
df = pd.read_csv("data/salaries.csv")

In [ ]:
df.head() # Mostrar las primeras 5 filas como vista previa de los datos

In [ ]:
df.info() # Mostrar información sobre el dataframe

Si nuestro objetivo es estimar el salario que tendrá un empleado en función de sus años de experiencia, consideraremos el número de años de experiencia como la **variable independiente** (también llamada predictor o feature) y el salario como la **variable dependiente** (también llamada target o variable de respuesta).

Esto significa que estamos modelando una **relación entre ambas variables**: asumimos que el salario depende (es función de) los años de experiencia del empleado. Nótese que, aunque nuestro modelo asume esta dirección de causalidad basándose en el conocimiento del dominio, la correlación por sí sola no prueba la causalidad — necesitamos comprensión teórica para justificar este supuesto.

## Visualización de datos

Crearemos un **diagrama de dispersión** para visualizar la relación entre ambas variables.

Un diagrama de dispersión muestra coordenadas cartesianas para mostrar los valores de dos variables para un dataset. En este caso, visualizaremos un punto por cada empleado, donde el eje X representa los años de experiencia y el eje Y representa su salario.

Utilizo el método `dataframe.plot` de **pandas** para crear el diagrama de dispersión. Este método, a su vez, usa la librería **matplotlib** para generar la visualización.

In [ ]:
df.plot(kind='scatter', # Crear un diagrama de dispersión
        x='YearsExperience', # Eje X (variable independiente)
        y='Salary', # Eje Y (variable dependiente)
        title='Salario en función de los años de experiencia',
        grid=True, # Mostrar cuadrícula
        xlabel='Años de Experiencia', # Etiqueta eje X
        ylabel='Salario') # Etiqueta eje Y

Podemos observar una clara **correlación** entre ambas variables: a mayor número de años de experiencia, mayor salario. Y la relación parece bastante lineal, lo que significa que podemos aproximarla con una línea recta (para un predictor; en multiple linear regression sería un plano o un **hiperplano**).

Los datos nos proporcionan una muestra de la realidad, y buscamos encontrar un **modelo** que la describa. Cuando luego apliquemos el modelo a un empleado del que solo conozcamos sus años de experiencia, obtendremos una **predicción** de su salario.

## Modelo de Linear Regression


Buscamos por tanto una línea recta que sea el modelo que mejor se ajuste a los datos. Sabiendo que una línea recta se define con la fórmula
$$\mathbf{y = m*x + b}$$
(donde $m$ es la pendiente de la recta y $b$ es la intersección con el eje y), necesitamos encontrar los valores de $m$ y $b$ que mejor se ajusten a los datos.

**El modelo estará formado por esta fórmula, y los parámetros del modelo son los coeficientes $\mathbf{m}$ y $\mathbf{b}$**.

In [ ]:
# Definir x como un vector con los datos de años de experiencia e y como un vector con los datos de salario correspondientes a cada valor de x
x = df['YearsExperience'].values # .values convierte esa columna del dataframe en un array de numpy
y = df['Salary'].values

Importamos la clase `linear_model.LinearRegression` de la librería **scikit-learn**. Esta clase implementa el **modelo** de linear regression y nos permite **entrenarlo** con los datos que tenemos.

Documentación sobre modelos lineales en scikit-learn: https://scikit-learn.org/stable/modules/linear_model.html

In [ ]:
from sklearn.linear_model import LinearRegression

El método `LinearRegression.fit()` recibe como parámetros los valores de las variables independiente y dependiente, y calcula los valores de los **parámetros del modelo** que mejor se ajustan a los datos.

In [ ]:
model = LinearRegression().fit(x.reshape(-1, 1), y) # Ajustar el modelo de linear regression

En este caso solo tenemos una variable independiente x (años de experiencia), pero podríamos tener más (**LinearRegression también soporta multiple linear regression**). Por eso el primer parámetro es una matriz bidimensional (un array de arrays) donde cada fila es una muestra y cada columna es una variable independiente. El parámetro X debe ser por tanto un array bidimensional donde cada fila es una muestra y cada columna es una variable independiente. Por eso usamos `reshape` para convertirlo en un array con 1 columna y tantas filas como muestras hay.

También podríamos pasar los datos directamente desde el dataframe; esto es equivalente al enfoque anterior pero incorpora los nombres de las features al modelo

In [ ]:
#model = LinearRegression().fit(df[['YearsExperience']], df['Salary'])

Es importante recordar que `df[['YearsExperience']]` devuelve un dataframe con una sola columna, mientras que `df['Salary']` devuelve una serie. Para la variable dependiente (y), el método acepta ambas opciones, aunque es más común pasar una serie dado que predice una única variable dependiente. Sin embargo, para las variables independientes (x), solo acepta datos en columnas ya que puede haber más de una variable independiente, en cuyo caso deberíamos pasar un dataframe con tantas columnas como variables independientes.
Este es el mismo motivo por el que, cuando usamos NumPy, utilizamos `reshape`.

In [ ]:
m = model.coef_[0] # Pendiente (coeficiente de x)
b = model.intercept_ # Intersección con el eje Y

# Con estos dos valores, aplicando la fórmula de la recta, podemos calcular el valor predicho para cualquier valor de x
ŷ = m * x + b

En este caso, usamos la notación ŷ para referirnos a las predicciones de la variable dependiente $y$. Es decir:
- ŷ es el valor que el modelo predice para un valor de $x$
- y son los valores reales de la variable dependiente (las muestras que tenemos desde el principio)

Esta notación es más matemática/estadística que específica de Python, pero es ampliamente utilizada en el campo de la ciencia de datos.

### Visualización de la línea de regresión y sus residuos

In [ ]:
 # Este gráfico es más completo que el anterior, por lo que uso la librería matplotlib directamente
from matplotlib import pyplot as plt

# Graficar los datos reales (y) y los datos estimados (ŷ) como puntos de dispersión
plt.scatter(x, y, c='blue', label='Datos reales (y)')
plt.scatter(x, ŷ, c='red', label='Datos estimados (ŷ)', alpha=0.25) # alpha es la transparencia de los puntos en la función scatter
# Graficar la línea de regresión
plt.plot(x, ŷ,  c='red', label=f'Línea de regresión ŷ = {round(m)}x + {round(b)}')

# Dibujar líneas de error (líneas desde los datos de entrada hasta la línea ajustada)
for xi, yi, ŷi in zip(x, y, ŷ): # zip() asocia elementos de las listas en tuplas, desempaquetamos e iteramos sobre cada trío de elementos
    plt.plot([xi, xi], [yi, ŷi], c='gray', linestyle='--', linewidth=.5) # dibuja una línea desde el punto (xi, yi) hasta el punto (xi, ŷi)

plt.xlabel('Años de Experiencia')  # Etiqueta eje X
plt.ylabel('Salario')  # Etiqueta eje Y
plt.title('Ajuste Lineal Simple en un Diagrama de Dispersión')
plt.legend() # Mostrar leyenda
plt.grid(True) # Añadir cuadrícula
plt.show() # Mostrar el gráfico

In [ ]:
x, y, ŷ

(array([ 2.3 ,  8.29,  5.  ,  5.4 ,  6.1 ,  1.2 ,  3.3 ,  4.1 ,  4.1 ,
         3.1 ,  4.  ,  4.6 ,  3.  ,  8.  , 10.4 ,  3.8 ,  6.9 ,  8.79,
         1.4 ,  9.1 ,  7.19, 10.6 ,  9.6 ,  4.19,  6.  ,  1.6 ,  5.19,
         3.3 ,  9.7 ,  2.1 ]),
 array([ 39892., 113813.,  67939.,  83089.,  93941.,  39344.,  54446.,
         56958.,  55795.,  60151.,  63219.,  61112.,  56643., 101303.,
        122392.,  57190.,  91739., 109432.,  46206., 105583.,  98274.,
        121873., 116970.,  57082.,  81364.,  37732.,  66030.,  64446.,
        112636.,  43526.]),
 array([ 46590.82024449, 103211.58513003,  72112.70091076,  75893.72026873,
         82510.50414517,  36193.01701008,  56043.3686394 ,  63605.40735533,
         63605.40735533,  54152.85896042,  62660.15251584,  68331.68155279,
         53207.60412093, 100470.34609551, 123156.46224331,  60769.64283686,
         90072.5428611 , 107937.85932749,  38083.52668906, 110868.14932992,
         92813.78189563, 125046.97192229, 115594.42352737,  6445

Podemos ver cada uno de los puntos junto con la línea de regresión que calculamos y el **residuo** de cada dato respecto a la predicción del modelo. Lo que hemos buscado es la línea que mejor minimiza estos **residuos**.

**Distinción importante:**
- **Residuo:** La diferencia entre un valor observado y el valor predicho por nuestro modelo (y - ŷ). Esto es lo que calculamos y visualizamos.
- **Error:** La diferencia entre un valor observado y el valor verdadero (desconocido), incluyendo el ruido de medición y los factores no controlados.

Podemos observar los residuos en nuestros datos, pero los errores verdaderos son desconocidos. Al entrenar, minimizamos los residuos, lo que equivale a minimizar el **mean squared error (MSE)** — el promedio de los residuos al cuadrado en todos los datos.

En la explicación anterior, extraemos los coeficientes del modelo y calculamos la fórmula de la recta con ellos para ayudar a entender el concepto, pero en realidad podríamos hacerlo directamente con el método `LinearRegression.predict()`, que devuelve las predicciones para los valores de entrada que le pasemos:

In [ ]:
plt.scatter(x, y, c='blue', label='Datos reales')
plt.plot(x, model.predict(x.reshape(-1, 1)), c='red', label=f'Línea de regresión')

plt.xlabel('Años de Experiencia')  # Etiqueta eje X
plt.ylabel('Salario')  # Etiqueta eje Y
plt.title('Ajuste Lineal Simple en un Diagrama de Dispersión')
plt.legend() # Mostrar leyenda
plt.grid(True) # Añadir cuadrícula
plt.show() # Mostrar el gráfico

## Métricas de error / Funciones de coste

Existen diferentes métricas para medir el error de un modelo de regresión. Las más comunes son el mean squared error (MSE) y el mean absolute error (MAE).

Estas métricas también se usan como **funciones de coste** a minimizar al entrenar el modelo.

### Mean Absolute Error (MAE)

Una forma de calcular el error es usando el **mean absolute error (MAE)**, que es la media de los errores absolutos cometidos al tomar $\hat y$ como predicción de $y$.

$$MAE(x,y) = \frac{1}{n}\sum_{i=1}^{n}|y_i-\hat y_i|$$

Esta métrica es la más intuitiva, ya que nos da una idea de cuánto se equivoca el modelo en cada predicción. Sin embargo, tiene un problema: requiere usar el valor absoluto de los errores para evitar que se cancelen entre sí (errores negativos restando a los positivos), ya que no nos importa si la predicción está por encima o por debajo del valor real — lo que nos importa es cuán lejos está. La función de valor absoluto no es diferenciable en 0, lo que dificulta su uso en **algoritmos de optimización**. El problema de calcular la línea que mejor se ajusta a los datos es un **problema de optimización**, ya que lo que buscamos es **minimizar la función de coste**.

In [ ]:
import numpy as np
mae = np.mean(np.abs(y - ŷ))
print("MAE calculado con su fórmula: ", mae)

from sklearn.metrics import mean_absolute_error
print("MAE calculado con scikit-learn: ", mean_absolute_error(y, ŷ))

### Mean Squared Error (MSE)

Debido a los problemas del MAE, es habitual usar el **mean squared error (MSE)**, que es la media de los errores al cuadrado.

$$ MSE(x,y)= \frac{1}{n}\sum_{i=1}^{n}{(y_i-\hat y_i)^2}$$

Al elevar al cuadrado los errores, ya evitamos que se cancelen entre sí. Además, al ser una función cuadrática, es diferenciable en todos los puntos, lo que facilita su uso en algoritmos de optimización.

Hay que tener en cuenta que al elevar al cuadrado penalizamos más los **outliers** (aquellos que están muy lejos de la predicción) que con el MAE. Esto puede ser o no deseable, dependiendo del problema. Este tipo de consideraciones abren la puerta a debatir cuál de las muchas métricas es más apropiada para cada problema.

Dado que el MSE es la media de los errores al cuadrado, el **RMSE** (root mean squared error) se usa frecuentemente como métrica de error para que tenga las mismas unidades que la variable dependiente, lo que facilita su interpretación.

$$ RMSE(x,y)= \sqrt{\frac{1}{n}\sum_{i=1}^{n}{(y_i-\hat y_i)^2}}$$

In [ ]:
from sklearn.metrics import mean_squared_error, root_mean_squared_error

print("MSE calculado con su fórmula: ", np.mean((y-ŷ)**2))
print("MSE calculado con scikit-learn: ", mean_squared_error(y, ŷ))

print("RMSE calculado con su fórmula: ", np.sqrt(np.mean((y-ŷ)**2)))
print("RMSE calculado con scikit-learn: ", root_mean_squared_error(y, ŷ))

### Ejemplo de errores en líneas no ajustadas

In [ ]:
# Generar dos líneas de ajuste erróneas para comparar con la línea de ajuste correcta
ŷ_err_1 = m*(1.15) * x + b - 5000 
ŷ_err_2 = m*(0.95) * x + b + 8000

plt.scatter(x, y, c='blue', label='Datos reales')
plt.plot(x, ŷ_err_1, c='red', label='Ajuste lineal erróneo #1')
plt.plot(x, ŷ_err_2, c='green', label='Ajuste lineal erróneo #2')
plt.xlabel('Años de Experiencia')  # Etiqueta eje X
plt.ylabel('Salario')  # Etiqueta eje Y
plt.title('Ajuste Lineal Simple en un Diagrama de Dispersión con Dos Líneas Erróneas')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
print(f"El MAE en la predicción #1 (errónea) es {round(mean_absolute_error(y,ŷ_err_1))} €")
print(f"El MAE en la predicción #2 (errónea) es {round(mean_absolute_error(y,ŷ_err_2))} €")
print(f"El MAE en la predicción correcta es {round(mean_absolute_error(y,ŷ))} €")

print(f"El MSE en la predicción #1 (errónea) es {round(mean_squared_error(y,ŷ_err_1))} €²")
print(f"El MSE en la predicción #2 (errónea) es {round(mean_squared_error(y,ŷ_err_2))} €²")
print(f"El MSE en la predicción correcta es {round(mean_squared_error(y,ŷ))} €²")

print(f"El RMSE en la predicción #1 (errónea) es {round(root_mean_squared_error(y, ŷ_err_1))} €")
print(f"El RMSE en la predicción #2 (errónea) es {round(root_mean_squared_error(y, ŷ_err_2))} €")
print(f"El RMSE en la predicción correcta es {round(root_mean_squared_error(y, ŷ))} €")

Podemos observar que el error en ambas métricas es mayor en las dos líneas erróneas que en la línea ajustada.

### Otras métricas

Existen otras métricas para medir el error de un modelo de regresión:
- **$R^2$** (coeficiente de determinación), que nos indica qué porcentaje de la variabilidad de la variable dependiente es explicado por el modelo. El valor de $R^2$ está entre 0 y 1, siendo 0 un modelo que no explica nada y 1 un modelo que explica toda la variabilidad de la variable dependiente.
- MAPE (Mean Absolute Percentage Error), que es el MAE expresado como porcentaje del valor real.
- SMAPE (Symmetric Mean Absolute Percentage Error), que es una versión simétrica del MAPE.
- Median Absolute Error, que es el MAE pero usando la mediana en lugar de la media.
- etc.

## Ajuste del modelo

Calculamos nuestro modelo con:
```python
from sklearn.linear_model import LinearRegression
model = LinearRegression().fit(x.reshape(-1, 1), y)
```

Esta función realizó el ajuste usando el método de **mínimos cuadrados ordinarios (OLS)**. Es una estrategia de optimización que busca minimizar la suma de los cuadrados de los residuos (la diferencia entre los valores reales y los predichos por el modelo).

Es decir, buscó los valores de $m$ y $b$ que **minimizan el mean squared error (MSE)**.

**Nota sobre OLS:** Para simple linear regression con un predictor, OLS tiene una solución analítica de forma cerrada — puede calcular los parámetros óptimos directamente usando álgebra matricial, sin optimización iterativa. Esto lo hace muy rápido de entrenar. Para modelos más complejos (como la regresión regularizada o la Logistic Regression), se necesitan algoritmos de optimización iterativos.

## Predicción de nuevos valores

Una vez que tenemos el modelo entrenado, podemos usarlo para predecir el salario de un empleado con un cierto número de años de experiencia.

In [ ]:
x_new = np.array([1.5, 2.5, 4.5, 5.5, 6.5, 10]) # Crear un vector con años de experiencia laboral para una serie de posibles nuevos empleados
ŷ_new = m * x_new + b # Calcular el salario estimado para cada uno de ellos

Normalmente, se calculará directamente usando el método `LinearRegression.predict()` de scikit-learn. Recuerda que fue con la clase `LinearRegression` que creamos el modelo y que los valores de m y b están en los atributos `coef_` e `intercept_` del objeto `model`.

In [ ]:
ŷ_new = model.predict(x_new.reshape(-1, 1))

In [ ]:
plt.scatter(x, y, c='blue', label='Datos reales')
plt.scatter(x_new, ŷ_new, c='red', label='Predicciones')
plt.xlabel('Años de Experiencia')
plt.ylabel('Salario')
plt.legend()
plt.grid(True)
plt.show()

## Multiple Linear Regression

Igual que con un solo predictor la linear regression se representa como una línea; con dos predictores se representaría como un plano; con tres predictores como un hiperplano, etc.

Aquí puedes ver una visualización de una linear regression con 2 predictores en 3D: https://miabellaai.net/regression.html

## De la Linear Regression a otros modelos de Machine Learning

La linear regression es uno de los modelos de machine learning más simples. En el caso de la simple linear regression, es un modelo con 2 **parámetros** (la pendiente y la intersección con el eje y). "Aprende" encontrando los valores de los parámetros que mejor se ajustan a los datos. Este "ajuste" se logra minimizando una función de coste específica; para la linear regression estándar, esta es la suma de los errores al cuadrado (las distancias verticales al cuadrado desde cada punto de datos hasta la línea).

Aunque la lógica subyacente y la arquitectura de otros modelos pueden ser muy diferentes, muchos comparten este proceso fundamental: usan un algoritmo de aprendizaje para ajustar sus parámetros internos para hacer que una función de pérdida específica sea lo más pequeña posible. La complejidad de este proceso es lo que escala dramáticamente. Por ejemplo, **un large language model (LLM) como GPT-3** tiene **175 mil millones de parámetros** a ajustar, en comparación con solo los dos de nuestra simple recta.

## Fuentes

- Este notebook está basado en [este](https://github.com/FranPuentes/iTI2024/blob/main/Cap.%2017%20-%20regresi%C3%B3n%20lineal%20simple.ipynb) de [Juan Francisco Puentes Calvo](https://github.com/FranPuentes/iTI2024)
- Se usa una versión modificada de [este dataset](https://www.kaggle.com/datasets/abhishek14398/salary-dataset-simple-linear-regression).
- [Linear Regression in Python](https://realpython.com/linear-regression-in-python)
- https://cienciadedatos.net/documentos/py10-regresion-lineal-python
- [Visualización de errores en linear regression](https://observablehq.com/@yizhe-ang/interactive-visualization-of-linear-regression)
- [Cost Function of Linear Regression: Deep Learning for Beginners](https://builtin.com/machine-learning/cost-function)